# 02 · Manual Frame Labelling

Keyboard-driven labelling tool. For each frame in `Frames/`, crops the top
200px (ceiling/background) and shows the result; typing one of `h`/`l`/`s`/`r`/`k`
writes a label to `labels.csv`. Progress is resumable — already-labelled
frames are skipped on re-run. `backwards`-direction frames are excluded
(collected as a robustness check, not used for training).

Labels: `h`=hard_left, `l`=left, `s`=straight, `r`=right, `k`=hard_right.
Any other key defaults to `straight`. `q` quits early.

Best run cell-by-cell in a local Jupyter session (needs a display).

In [ ]:
import cv2
import glob
import csv
import os
from matplotlib import pyplot as plt
from IPython.display import clear_output

# Get all images recursively
images = sorted(glob.glob("Frames/**/*.jpg", recursive=True))

# Remove backwards run
images = [img for img in images if "backwards" not in img.lower()]

print("Total images found:", len(images))

# Resume from last labelled image
labeled_images = set()

if os.path.exists("labels.csv"):
    with open("labels.csv", "r") as f:
        reader = csv.reader(f)
        next(reader, None)  # skip header
        for row in reader:
            labeled_images.add(row[0])

# Remove already labelled images
images = [img for img in images if img not in labeled_images]

print("Remaining images to label:", len(images))

# Open CSV for writing (append mode)
csv_file = open("labels.csv", "a", newline="")
writer = csv.writer(csv_file)

# Only write header if file is empty
if os.stat("labels.csv").st_size == 0:
    writer.writerow(["image", "label"])

In [ ]:
label_map = {
    "h": "hard_left",
    "l": "left",
    "s": "straight",
    "r": "right",
    "k": "hard_right",
}

for i, img_path in enumerate(images):

    img = cv2.imread(img_path)

    if img is None:
        print(f"Failed: {img_path}")
        continue

    if img.shape[0] > 200:
        img = img[200:, :]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    plt.title(f"Frame {i+1}/{len(images)} | h/l/s/r/k = label, q = quit")
    plt.axis("off")
    plt.show()

    key = input("Label this frame (h/l/s/r/k, q=quit): ").strip().lower()

    clear_output(wait=True)

    if key == "q":
        break
    elif key in label_map:
        writer.writerow([img_path, label_map[key]])
        print(f"{img_path} -> {label_map[key]}")
    else:
        writer.writerow([img_path, "straight"])
        print(f"{img_path} -> straight")

In [ ]:
cv2.destroyAllWindows()
csv_file.close()
print("Labelling complete! Saved to labels.csv")

## Sanity check: label distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("labels.csv")
df["label"].value_counts().plot(kind="bar")
plt.title("Label Distribution")
plt.show()